### **Groupby Object in Pandas**
- Help in making group of data from dataset columns
- Always applies on categorical data
- Make group by dividing data of given columns

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
movies = pd.read_csv('Pandas_Dataset\\imdb-top-1000.csv')
movies.head(1)

In [ ]:
# groupby
genres = movies.groupby('Genre')

In [ ]:
# Applying built-in aggregation functions on groupby objects
genres.sum()

- Groupby applied on Genre column made Dataframe where Genre values are index.
- Applying sum on that Groupby object gives the aggregated value of all the remaining columns of the dataset.

In [ ]:
genres.min().head()

In [ ]:
# Find the top 3 genres by total earning
movies.groupby('Genre').sum()['Gross'].sort_values(ascending=False).head(3)

In [ ]:
movies.groupby('Genre')['Gross'].sum().sort_values(ascending=False).head(3) 
# Other and best way of writing because sum() is not applying on whole dataset but only on Genre here

In [ ]:
# Find the genre with highest avg IMDB rating
movies.groupby('Genre')['IMDB_Rating'].mean().sort_values(ascending=False).head(1)

In [ ]:
# Find director with most popularity (number of votes)
movies.groupby('Director')['No_of_Votes'].sum().sort_values(ascending=False).head(1)

In [ ]:
# Find number of movies done by each actor
movies.groupby('Star1')['Series_Title'].count().sort_values(ascending=False)

### Groupby Attributes and Methods

In [ ]:
# Length of groups formed
len(genres)

In [ ]:
movies['Genre'].nunique()

In [ ]:
# Number of rows in each group
genres.size()

In [ ]:
movies['Genre'].value_counts()

In [ ]:
# First data of every group
genres.first()

In [ ]:
# Last data of every group
genres.last()

In [ ]:
# Any nth position data of groups
genres.nth(4)

In [ ]:
# Get data of specific groups
genres.get_group('Horror')

In [ ]:
movies[movies['Genre'] == 'Horror']

In [ ]:
# Dictionary of every groups with there indices values
genres.groups

In [ ]:
# describe() on groupby object
genres.describe()

In [ ]:
# sample() on groupby object
genres.sample()

In [ ]:
genres.sample(2, replace=True) # Duplicate values if data values are less then reequired values

In [ ]:
# nunique() on groupby object
genres.nunique()

### Aggregate Methods

In [ ]:
# Passing dict
genres.agg({
    'Runtime':'mean',
    'IMDB_Rating':'mean',
    'No_of_Votes':'sum',
    'Gross':'sum',
    'Metascore':'min'
})

In [ ]:
# Passing list
genres[movies.select_dtypes('number').columns].agg(['min','max','mean'])

In [ ]:
# Using both list and dict
genres.agg({
    'Runtime':['min','mean'],
    'IMDB_Rating':'mean',
    'No_of_Votes':['sum','max'],
    'Gross':'sum',
    'Metascore':'min'
})

### Looping on Groups

In [ ]:
for group, data in genres:
    print(type(group), type(data))

In [ ]:
for group, data in genres:
    print(group)

In [ ]:
for group, data in genres:
    print(data)

In [ ]:
# Find highest rated movie of each genre
df = pd.DataFrame(columns=movies.columns) # Empty dataframe with same columns as of movies dataset

for group, data in genres:
    df = pd.concat([df, data[data['IMDB_Rating'] == data['IMDB_Rating'].max()]])

df

### Split (Apply) Combine Strategy
- **Split** data by using groupby()
- Use **apply()** function to provide transformation to each group
- After transformation, **Combine** the result.

In [ ]:
# Find number of movies starting with A for each genre
genres['Series_Title'].apply(lambda x: x.str.startswith('A').sum())

In [ ]:
def startsA(group):
    return group['Series_Title'].str.startswith('A').sum()

genres.apply(startsA)

In [ ]:
# Find ranking of each movie in each genre according to IMDB score
def rank_movies(group):
    group['Genre_Rank'] = group['IMDB_Rating'].rank(ascending=False)
    return group

In [ ]:
genres.apply(rank_movies)

In [ ]:
# Find normalized IMDB rating genre wise
def normal(group):
    group['Norm_Rating'] = (group['IMDB_Rating'] - group['IMDB_Rating'].min())/(group['IMDB_Rating'].max() - group['IMDB_Rating'].min())
    return group

genres.apply(normal)

### Groupby on multiple columns

In [ ]:
duo = movies.groupby(['Director','Star1'])
duo.size()

In [ ]:
duo.get_group(('Aamir Khan','Amole Gupte'))

In [ ]:
# Find the most earning actor-director combo
duo['Gross'].sum().sort_values(ascending=False).head(1)

In [ ]:
# Find the best avg metascore of actor-genre combo
movies.groupby(['Star1','Genre'])['Metascore'].mean().sort_values(ascending=False).head(1)

In [ ]:
# Agg function on multiple groupby
duo[movies.select_dtypes('number').columns].agg(['min','max','mean'])

### Examples

In [ ]:
ipl = pd.read_csv('Pandas_Dataset\\deliveries.csv')
ipl.head()

In [ ]:
# Find the top 10 batsman in terms of runs
ipl.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False).head(10)

In [ ]:
# Find the batsman with max number of sixes
six = ipl[ipl['batsman_runs'] == 6]
six.groupby('batsman')['batsman'].count().sort_values(ascending=False).head(1)

In [ ]:
# Find batsman with most number of 4's and 6's in last 5 overs
new_df = ipl[((ipl['batsman_runs'] == 6) | (ipl['batsman_runs'] == 4)) & (ipl['over'] > 15)]
new_df.groupby('batsman')['batsman'].count().sort_values(ascending=False).head(1)

In [ ]:
# Find Virat Kohli's runs against all teams in every match
df = ipl[ipl['batsman'] == 'V Kohli']
df.groupby(['bowling_team','match_id'])['batsman_runs'].sum().reset_index()

In [ ]:
# Create a function that can return the highest score of any batsman
def highest(batsman):
    temp_df = ipl[ipl['batsman'] == batsman]
    return temp_df.groupby('match_id')['batsman_runs'].sum().sort_values(ascending=False).head(1)

highest('DA Warner')